In [1]:
import sys
sys.path.append('..')
from breastclip.model.modules import load_image_encoder, LinearClassifier, load_text_encoder, load_projection_head
from breastclip.data.data_utils import load_tokenizer
from Classifiers.models.breast_clip_classifier import BreastClipClassifier
import torch
import os
from ruamel.yaml import YAML
import pandas as pd


/home/ixb004/miniconda3/envs/mammoclip/lib/python3.8/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
INFO:albumentations.check_version:A new version of Albumentations is available: 2.0.2 (you have 1.4.7). Upgrade using: pip install --upgrade albumentations


Load Checkpoint

In [10]:
class Args:
    def __init__(self):
        self.device = 'cuda' if torch.cuda.is_available() else 'cpu'
        self.clip_chk_pt_path = '/mnt/PURENFS/SalkowskiPreprocessedBreast/code/MammoCLIP/checkpoints/b5-model-best-epoch-7.tar'
# Create an instance of the Args class
args = Args()
ckpt = torch.load(args.clip_chk_pt_path, map_location="cpu")
ckpt["config"]["tokenizer"]['cache_dir'] = "/home/ixb004/.cache/huggingface/hub/"
ckpt["config"]["model"]["text_encoder"]["cache_dir"] = "/home/ixb004/.cache/huggingface/hub/"
ckpt["config"]["model"]["text_encoder"]["name"] = "/home/ixb004/.cache/huggingface/hub/models--emilyalsentzer--Bio_ClinicalBERT/"

Load Image encoder

In [12]:
image_encoder = load_image_encoder(ckpt['config']['model']['image_encoder'])

<All keys matched successfully>
Loaded pretrained weights for efficientnet-b5


Load Text Encoder

In [14]:
tokenizer = load_tokenizer(**ckpt["config"]["tokenizer"])
text_encoder = load_text_encoder(ckpt["config"]["model"]["text_encoder"], vocab_size=tokenizer.vocab_size)

/home/ixb004/miniconda3/envs/mammoclip/lib/python3.8/site-packages/huggingface_hub/file_download.py:1132: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


Load UW Data

In [2]:
sys.path.insert(0,'/mnt/PURENFS/SalkowskiPreprocessedBreast/code/ALBEF')
from dataset import create_dataset, create_sampler, create_loader

In [52]:
tr = pd.read_json('data/uw_ten_shot_set.json')
val = pd.read_json('data/uw_madison_val.json')
test = pd.read_json('data/uw_madison_test_with_groupids.json')
train_all = pd.read_json('data/uw_madison_train.json')
grpid2grp = pd.read_pickle('data/groupid2grp.pkl')
vc = train_all.group_id.value_counts()
rare_ids = set(vc[20:].index)
freq_ids = set(vc[:20].index)
# Convert group_id columns to unique sets
test_ungrp = set(test.group_id.unique())
tr_ungrp = set(tr.group_id.unique())
val_ungrp = set(val.group_id.unique())
print("Number of Rare and freq grps",len(rare_ids),len(freq_ids))
print("Train, val,test df shapes ", tr.shape,val.shape,test.shape, train_all.shape)
# Print the lengths of each set to show the total number of unique group IDs
print(f"Unique groups in val: {len(val_ungrp)}")
print(f"Unique groups in tr: {len(tr_ungrp)}")
print(f"Unique groups in test: {len(test_ungrp)}")

# Union: Combine all unique groups from the three sets
union_groups = val_ungrp.union(tr_ungrp, test_ungrp)
print(f"Union of all groups (val, tr, test): {len(union_groups)}")

# Intersection of different pairs: Find common groups between each pair of sets
intersection_val_tr = val_ungrp.intersection(tr_ungrp)
intersection_val_test = val_ungrp.intersection(test_ungrp)
intersection_test_tr = test_ungrp.intersection(tr_ungrp)

# Print the sizes of intersections
print(f"Intersection of val and tr: {len(intersection_val_tr)}")
print(f"Intersection of val and test: {len(intersection_val_test)}")
print(f"Intersection of test and tr: {len(intersection_test_tr)}")

# Intersection of all three sets: Find common groups across val, tr, and test
intersection_all = val_ungrp.intersection(tr_ungrp).intersection(test_ungrp)
print(f"Intersection of all three sets (val, tr, test): {len(intersection_all)}")

# Detailed set differences: Groups that are in one set but not in others
# Groups in val and test but NOT in train
val_test_not_train = (val_ungrp.intersection(test_ungrp)) - tr_ungrp
print(f"Groups in both val and test, but NOT in train: {len(val_test_not_train)}")
print(f"Example groups: {list(val_test_not_train)}")

# Groups in val but NOT in train or test
val_not_train_test = val_ungrp - tr_ungrp - test_ungrp
print(f"Groups in val but NOT in train or test: {len(val_not_train_test)}")
print(f"Example groups: {list(val_not_train_test)}")

# Groups in test but NOT in val or train
test_not_val_train = test_ungrp - val_ungrp - tr_ungrp
print(f"Groups in test but NOT in val or train: {len(test_not_val_train)}")
print(f"Example groups: {list(test_not_val_train)}")
print(f" Names groups : {[grpid2grp[i] for i in test_not_val_train]}")

# Groups in train but NOT in val or test
train_not_val_test = tr_ungrp - val_ungrp - test_ungrp
print(f"Groups in train but NOT in val or test: {len(train_not_val_test)}")
print(f"Example groups: {list(train_not_val_test)}")

print("Rare ids in Train , Val and Test", len(rare_ids.intersection(tr_ungrp)), len(rare_ids.intersection(test_ungrp)), len(rare_ids.intersection(val_ungrp)))
print("Freq ids in Train , Val and Test", len(freq_ids.intersection(tr_ungrp)), len(freq_ids.intersection(test_ungrp)), len(freq_ids.intersection(val_ungrp)))

Number of Rare and freq grps 985 20
Train, val,test df shapes  (3331, 6) (1000, 4) (72328, 4) (70328, 6)
Unique groups in val: 105
Unique groups in tr: 1005
Unique groups in test: 1017
Union of all groups (val, tr, test): 1017
Intersection of val and tr: 100
Intersection of val and test: 105
Intersection of test and tr: 1005
Intersection of all three sets (val, tr, test): 100
Groups in both val and test, but NOT in train: 5
Example groups: [1005, 1006, 1007, 1008, 1009]
Groups in val but NOT in train or test: 0
Example groups: []
Groups in test but NOT in val or train: 7
Example groups: [1010, 1011, 1012, 1013, 1014, 1015, 1016]
 Names groups : [('round, benign calcification', 'round, obscured, oval, circumscribed', 'fatty'), ('reduction', 'scattered fibroglandular densities', 'oval, high density', 'benign calcification'), ('scattered fibroglandular densities', 'architectural distortion', 'round, circumscribed'), ('lymph node', 'fatty', 'lumpectomy'), ('biopsy clip implant', 'architect

In [6]:
script_path = '/mnt/PURENFS/SalkowskiPreprocessedBreast/code/ALBEF/'
os.chdir(script_path)
# config_path = '/mnt/PURENFS/SalkowskiPreprocessedBreast/code/ALBEF/configs/Pretrain.yaml'
config_path = '/mnt/PURENFS/SalkowskiPreprocessedBreast/code/ALBEF/configs/Retrieval_coco.yaml'
yaml = YAML(typ='safe')  # 'safe' for safe loading
with open(config_path, 'r') as file:
    config = yaml.load(file)
# pre_dataset = create_dataset('pretrain', config)
TOP_N = 500
config['normalize'] = { "mean": 0.3089279  ,"std": 0.25053555408335154}
train_dataset, val_dataset, test_dataset = create_dataset('re', config)

samplers = [None]*3
data_loader = create_loader([train_dataset,test_dataset,val_dataset],samplers,
                            batch_size=[config['batch_size_test']]*3,
                            num_workers=[4]*3,
                            is_trains=[False]*3, 
                            collate_fns=[None]*3) 
# data_loader = create_loader([pre_dataset],samplers,batch_size=[config['batch_size_train']], num_workers=[4], is_trains=[True], collate_fns=[None])[0]

created SelectiveSampling object.


100%|██████████| 3331/3331 [00:00<00:00, 1557029.60it/s]


Dataset size:3331


1000it [00:00, 55367.43it/s]

Dataset size:1000



72328it [00:01, 48093.91it/s]

Dataset size:72328


In [ ]:
train_dataset.__getitem__(0)

In [7]:
for image,text,_ in data_loader[0]:
    print(image.shape,len(text))

    print(image[0].mean(),image[0].std(),image[1].min(),image[0].max())
    
    break
for image,text in data_loader[1]:
    print(image.shape,len(text))

    print(image[0].mean(),image[0].std(),image[1].min(),image[0].max())
    
    break

torch.Size([8, 3, 512, 512]) 8
tensor(-0.0374) tensor(1.1387) tensor(-1.2331) tensor(2.7584)
torch.Size([8, 3, 512, 512]) 8
tensor(-0.3455) tensor(0.8517) tensor(-1.2331) tensor(2.7584)


In [5]:

for image,text,_ in data_loader[0]:
    print(image.shape,len(text))

    print(image[0].mean(),image[0].std(),image[1].min(),image[0].max())
    
    break
for image,text in data_loader[1]:
    print(image.shape,len(text))

    print(image[0].mean(),image[0].std(),image[1].min(),image[0].max())
    
    break


torch.Size([8, 3, 512, 512]) 8
tensor(-0.3455) tensor(0.8517) tensor(-1.2331) tensor(2.7584)
torch.Size([8, 3, 512, 512]) 8
tensor(-0.3455) tensor(0.8517) tensor(-1.2331) tensor(2.7584)
